In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
class Expert(nn.Module):
    """A standard feedforward network that serves as an Expert."""
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.ffl1 = nn.Linear(in_dim, hidden_dim)
        self.ffl2 = nn.Linear(hidden_dim, in_dim)
    
    def forward(self, x):
        out1 = self.ffl1(x)
        out2 = self.ffl2(F.silu(out1))

        return out2


In [30]:
class MoELayer(nn.Module):
    """The Mixture of Experts Layer."""
    def __init__(self, in_dim, hidden_dim, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        self.router = nn.Linear(in_dim, num_experts, bias=False)

        self.experts = nn.ModuleList(
            [ Expert(in_dim, hidden_dim) for _ in range(self.num_experts)]
        )
    
    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model) # (batch_size x seq_en, d_model)
        
        # (batch_size x seq_en, d_model) @ (in_dim, num_experts) = (batch_size x seq_en, num_experts)
        # Output Shape: (batch_size x seq_en, num_experts)
        # Every input token has a list of expert probability scores.
        # Meaning, if we do a softmax, we can select top K experts for
        # each input token and direct them to those experts.
        router_logits = self.router(x_flat)
        routing_probs = F.softmax(router_logits, dim=-1)

        # Selecting the top_k experts for each input token.
        top_k_probs, top_k_indices = torch.topk(routing_probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        print(f'Routing Probs: {routing_probs}')
        print(f'Top K Indices: {top_k_indices}')
        print(f'Top K Probs: {top_k_probs}')

        final_output = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            # Create a boolean mask for tokens assigned to the current expert (i)
            expert_mask = (top_k_indices == i)
            print(f'Expert Mask: {expert_mask}')

            # Get the flat indices of these tokens
            token_indices = expert_mask.any(dim=-1)

            if not token_indices.any():
                # Skip if no tokens were routed to this expert
                continue

            # Extract the specific tokens and pass them through the expert
            token_for_expert = x_flat[token_indices] # (num_tokens_having_the_expert_in_topk, in_dim)
            expert_output = expert(token_for_expert) # (num_tokens_having_the_expert_in_topk, in_dim)
            print(f'Parent Tensor shape: {x_flat.shape}')
            print(f'Token for Expert shape: {token_for_expert.shape}')
            print(f'Expert Output Shape: {expert_output.shape}')
            print(f'Selected Tokens for this Expert: {token_indices}')

            # Extract the corresponding normalized routing probabilities
            # .nonzero(as_tuple=True)[1] tells us if this expert was the 1st or 2nd choice
            weight_indices = expert_mask[token_indices].nonzero(as_tuple=True)[1]
            print(f'Weight Indices: {weight_indices}')
            token_weights = top_k_probs[token_indices, weight_indices].unsqueeze(-1)
            print(f'Weights: {token_weights} and {token_weights.shape}')

            # Weight the expert's output and add it to the final result
            final_output[token_indices] += expert_output * token_weights
            print(f'Overall Weights: {final_output}')
            print(f'Final Output Weights: {final_output[token_indices]}')
            print('\n\n')
        
        return final_output.view(batch_size, seq_len, d_model)

In [31]:
d_model, hidden_dim = 16, 32
num_experts, top_k = 8, 2

# Create random dummy text data (Batch: 2, Sequence Length: 4)
x = torch.randn(2, 4, d_model)

# Initialize our MoE layer
moe_layer = MoELayer(d_model, hidden_dim, num_experts, top_k)

# Run the forward pass
output = moe_layer(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

Routing Probs: tensor([[0.1647, 0.2128, 0.0744, 0.2912, 0.0412, 0.0329, 0.0757, 0.1070],
        [0.0960, 0.2000, 0.1028, 0.0741, 0.1144, 0.1191, 0.2129, 0.0806],
        [0.1033, 0.1057, 0.0762, 0.1992, 0.0677, 0.1146, 0.2088, 0.1245],
        [0.1287, 0.1532, 0.1222, 0.0479, 0.1271, 0.1115, 0.1974, 0.1119],
        [0.1681, 0.1665, 0.1028, 0.1575, 0.1224, 0.1117, 0.1215, 0.0495],
        [0.0895, 0.0835, 0.0957, 0.1028, 0.2085, 0.1091, 0.1732, 0.1378],
        [0.0850, 0.0913, 0.1798, 0.2140, 0.0672, 0.1095, 0.1616, 0.0915],
        [0.1408, 0.1750, 0.1115, 0.1075, 0.1435, 0.1521, 0.1016, 0.0679]],
       grad_fn=<SoftmaxBackward0>)
Top K Indices: tensor([[3, 1],
        [6, 1],
        [6, 3],
        [6, 1],
        [0, 1],
        [4, 6],
        [3, 2],
        [1, 5]])
Top K Probs: tensor([[0.5778, 0.4222],
        [0.5156, 0.4844],
        [0.5118, 0.4882],
        [0.5630, 0.4370],
        [0.5024, 0.4976],
        [0.5463, 0.4537],
        [0.5434, 0.4566],
        [0.5349, 0